# 3. Preparación del dataset para clustering

Este notebook construye la matriz de clustering lista para modelado:
imputación de missings, winsorización, transformación log(x+1) y 
estandarización z-score. La salida es `B01_MATRIZ_CLUSTERING.parquet` 
— una fila por contribuyente, 68 variables estandarizadas.

In [1]:
import sys, os
ROOT = r"D:\inf_sri_hist3\z_CLUSTER"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

import config
import importlib
importlib.reload(config)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    "figure.dpi": 150, "figure.figsize": (10, 5),
    "axes.spines.top": False, "axes.spines.right": False,
})

# Cargar dataset integrado
df = pd.read_parquet(config.PARQUET_FILE)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Núcleo de clustering: {len(config.CLUSTER_VARS_FINAL)} variables")

Dataset cargado: 797,161 filas × 263 columnas
Núcleo de clustering: 68 variables


## 3.1 Extracción del núcleo de clustering y registro de missings pre-imputación

In [2]:
# Extraer solo las 68 variables del núcleo + ID
X = df[["ID"] + config.CLUSTER_VARS_FINAL].copy()

# Registro de missings antes de imputar — para documentación
miss_pre = (X[config.CLUSTER_VARS_FINAL]
            .isnull().sum()
            .reset_index()
            .rename(columns={"index": "variable", 0: "n_missing"}))
miss_pre["pct"] = (miss_pre["n_missing"] / len(X) * 100).round(2)
miss_pre = miss_pre[miss_pre["n_missing"] > 0].sort_values("pct", ascending=False)

print(f"Variables con missing antes de imputar: {len(miss_pre)} de {len(config.CLUSTER_VARS_FINAL)}")
print(f"Total celdas con missing: {X[config.CLUSTER_VARS_FINAL].isnull().sum().sum():,}")

Variables con missing antes de imputar: 31 de 68
Total celdas con missing: 5,516,674


## 3.2 Imputación de missings

Estrategia diferenciada según el origen semántico del missing:
- **Imputar cero**: ausencia equivale a comportamiento nulo 
  (sin retenciones, sin recaudación, sin demora registrada).
- **Imputar mediana**: valor desconocido, no necesariamente cero
  (variables de F02 con missing por ausencia de declaración de renta).

In [3]:
# Variables que se imputan con CERO
# (ausencia = comportamiento nulo, no valor desconocido)
imputa_cero = [v for v in config.CLUSTER_VARS_FINAL if v not in config.F02_VARS]

# Variables F02 que se imputan con MEDIANA
# (sin declaración de renta = valor desconocido)
imputa_mediana = [v for v in config.F02_VARS if v in config.CLUSTER_VARS_FINAL]

# Imputación cero
for v in imputa_cero:
    n_antes = X[v].isnull().sum()
    if n_antes > 0:
        X[v] = X[v].fillna(0)

# Imputación mediana
medianas = {}
for v in imputa_mediana:
    med = X[v].median()
    medianas[v] = med
    n_antes = X[v].isnull().sum()
    if n_antes > 0:
        X[v] = X[v].fillna(med)

# Verificación
n_miss_post = X[config.CLUSTER_VARS_FINAL].isnull().sum().sum()
print(f"Missing tras imputación: {n_miss_post}")
print(f"\nMedianas usadas para F02:")
for v, med in medianas.items():
    print(f"  {v:<30s} mediana = {med:.4f}")

Missing tras imputación: 0

Medianas usadas para F02:
  I_ingresos_avg                 mediana = 17354.0767
  margen_operativo_avg           mediana = 0.3558
  carga_efectiva_ir_avg          mediana = 0.0079
  gap_ret_ir_avg                 mediana = 0.4194
  prop_anios_perdida             mediana = 0.0000


## 3.3 Winsorización

Se aplican los umbrales confirmados en el EDA:
- Grupo B (ratios con denominador cercano a cero): P99 calculado en EDA
- F04 ratios NC: P99.5 calculado en EDA

In [4]:
# Winsorización Grupo B — ratios con denominador cercano a cero
n_winsor = {}
for v, umbral in config.WINSOR_ANTES_LOG.items():
    if v in X.columns:
        n_afect = (X[v] > umbral).sum()
        X[v] = X[v].clip(upper=umbral)
        n_winsor[v] = n_afect

# Winsorización F04 ratios NC
for v, umbral in config.WINSOR_PARAMS.items():
    if v in X.columns:
        n_afect = (X[v] > umbral).sum()
        X[v] = X[v].clip(upper=umbral)
        n_winsor[v] = n_afect

print("Winsorización aplicada:")
print(f"  {'Variable':<30s} {'Umbral':>12s} {'N afectados':>12s} {'% universo':>10s}")
print("  " + "-" * 68)
umbrales_todos = {**config.WINSOR_ANTES_LOG, **config.WINSOR_PARAMS}
for v, n in n_winsor.items():
    print(f"  {v:<30s} {umbrales_todos[v]:>12.4f} {n:>12,} {n/len(X)*100:>9.3f}%")

Winsorización aplicada:
  Variable                             Umbral  N afectados % universo
  --------------------------------------------------------------------
  ratio_pago_causado                  96.7939        6,608     0.829%
  ratio_nc_fe                          0.3167        3,962     0.497%
  ratio_nc_recibidas_fe                0.4525        3,984     0.500%


## 3.4 Transformación log(x+1)

In [5]:
# Aplicar log(x+1) a variables del Grupo A
log_vars_en_nucleo = [v for v in config.LOG_VARS_EN_CLUSTER
                      if v in X.columns]

for v in log_vars_en_nucleo:
    X[v] = np.log1p(X[v].clip(lower=0))

# Verificar que no quedaron infinitos o NaN
n_inf = np.isinf(X[config.CLUSTER_VARS_FINAL]).sum().sum()
n_nan = X[config.CLUSTER_VARS_FINAL].isnull().sum().sum()
print(f"Tras log(x+1):")
print(f"  Variables transformadas : {len(log_vars_en_nucleo)}")
print(f"  Infinitos generados     : {n_inf}")
print(f"  NaN generados           : {n_nan}")
print(f"  Estado                  : {'OK' if n_inf == 0 and n_nan == 0 else 'REVISAR'}")

Tras log(x+1):
  Variables transformadas : 28
  Infinitos generados     : 0
  NaN generados           : 0
  Estado                  : OK


## 3.5 Estandarización z-score

In [6]:
# Estandarización z-score sobre las 68 variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[config.CLUSTER_VARS_FINAL])
X_scaled = pd.DataFrame(X_scaled,
                         columns=config.CLUSTER_VARS_FINAL,
                         index=X.index)

# Verificación — media ~0, std ~1
medias  = X_scaled.mean().round(6)
desv    = X_scaled.std().round(6)

print("Verificación post-estandarización:")
print(f"  Media máxima absoluta  : {medias.abs().max():.6f}  (esperado ≈ 0)")
print(f"  Std mínima             : {desv.min():.6f}  (esperado ≈ 1)")
print(f"  Std máxima             : {desv.max():.6f}  (esperado ≈ 1)")
print(f"  Shape matriz final     : {X_scaled.shape}")

Verificación post-estandarización:
  Media máxima absoluta  : 0.000000  (esperado ≈ 0)
  Std mínima             : 1.000001  (esperado ≈ 1)
  Std máxima             : 1.000001  (esperado ≈ 1)
  Shape matriz final     : (797161, 68)


## 3.6 Guardado de la matriz de clustering

In [7]:
# Añadir ID para trazabilidad
X_final = X_scaled.copy()
X_final.insert(0, "ID", X["ID"].values)

# Ruta de salida
MATRIZ_FILE = config.DATA_DIR / "B01_MATRIZ_CLUSTERING.parquet"

X_final.to_parquet(MATRIZ_FILE, engine="pyarrow",
                   compression="snappy", index=False)

size_mb = MATRIZ_FILE.stat().st_size / 1e6
print(f"Matriz de clustering guardada:")
print(f"  Ruta    : {MATRIZ_FILE}")
print(f"  Shape   : {X_final.shape[0]:,} filas × {X_final.shape[1]} columnas")
print(f"  Tamaño  : {size_mb:.1f} MB")
print(f"\nEl siguiente notebook es 04_clustering.ipynb")

Matriz de clustering guardada:
  Ruta    : D:\inf_sri_hist3\z_CLUSTER\data\B01_MATRIZ_CLUSTERING.parquet
  Shape   : 797,161 filas × 69 columnas
  Tamaño  : 196.8 MB

El siguiente notebook es 04_clustering.ipynb
